In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# Filtering Practice

시간영역 신호에서 원하는 주파수 대역만 추려내거나, 불필요한 저주파 드리프트 · 고주파 잡음을 제거하는 실습.

모든 필터는 `utils.filtering(x, fs, ftype, ...)` 하나로 통합되어 있으며 내부적으로 `scipy.signal.butter` + `filtfilt` (zero-phase) 로 구현되어 있다.

**Part A — 기본 개념** (3·10·15 Hz 사인파 + 잡음, 본문 §2.3 와 1:1 일치):
- 실습 1. 주파수 분석으로 신호 구성 확인하기
- 실습 2. Band-pass 필터로 관심 대역 추출하기 (causal vs zero-phase 비교)
- 실습 3. High-pass 필터로 저주파 사인파 제거
- 실습 4. Low-pass 필터로 고주파 사인파 제거
- 실습 5. High + Low ≈ Original (상보성 관찰)

**Part B — 응용 시나리오** (drift + 공진 + 잡음, 본문 §2.4 와 1:1 일치):
- 실습 6. 응용 신호 — drift + 공진 + 잡음 (raw 시간 + 스펙트럼)
- 실습 7. HPF 로 drift 제거
- 실습 8. BPF 로 공진 대역 추출
- 실습 9. 신호 분해 — drift / 공진 / 잡음 3 성분 분리

## 준비: 라이브러리 & 데이터 로드

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import fft, filtering, filtering_zerophase

In [ ]:
fs = 1000  # sampling frequency (Hz)

data = np.array(pd.read_csv('./data/data_sample_filtering.csv'))
print('shape:', data.shape)

t = data[:, 1]
v = data[:, 2]

---

## 실습 1. 주파수 분석으로 신호 구성 확인하기

필터를 설계하기 전에 **어떤 주파수 성분이 얼마나 섞여 있는가** 를 먼저 파악한다.
`utils.fft(v, fs)` 는 단측 진폭 스펙트럼 `(f, A)` 를 반환한다.

상단은 시간영역, 하단은 0–30 Hz 확대한 주파수영역을 함께 그려 공진 대역과 저주파 드리프트의 존재를 확인한다.

In [ ]:
f, A = fft(v, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v, 'C0')
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].set_title('Raw signal (time domain)')

ax[1].plot(f, A, 'C0')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 30])
ax[1].set_ylim([0, 1.2])
ax[1].set_title('Single-sided amplitude spectrum (0 to 30 Hz)')

fig.tight_layout()
plt.show()

**관찰.** 0 Hz 근처의 DC/저주파 성분과 5–12 Hz 부근의 공진 피크가 함께 보인다. 즉 이 신호는 느린 드리프트 + 관심 대역 진동 + 고주파 잡음이 혼재된 상태다. 이후 실습에서는 이 세 성분을 각각 어떻게 분리할 수 있는지 확인한다.

---

## 실습 2. Band-pass 필터로 관심 대역 추출하기

`utils.filtering` 은 두 가지 방식을 제공한다.

- `filtering(x, fs, ftype, ...)` — **인과(causal) 단일 패스** (`sosfilt` 기반).
  실시간 시스템에 적용 가능하지만 **위상 지연** 이 누적된다.
- `filtering_zerophase(x, fs, ftype, ...)` — **영위상(zero-phase)** (`sosfiltfilt` 기반).
  신호를 앞뒤로 두 번 필터링해 위상 지연을 상쇄. 사후 분석 전용(비인과).
  크기 응답이 $|H|^2$ 가 되어 전이대가 더 가파르다.

두 방식을 같은 Band-pass 에 적용해 **위상 차이** 를 확인한다.

> **실무 팁**: 사후 분석에는 `filtering_zerophase` 를 기본으로 쓴다. 계수 차수가 높거나 cutoff 가 Nyquist 에 매우 가까워 `filtering_zerophase` 가 수치적으로 발산하면 `filtering` (causal) 으로 우회한다. 이 노트북 이후의 모든 실습은 `filtering_zerophase` 를 사용한다.

In [ ]:
v_bp_causal = filtering(v,           fs, 'band', f_low=5, f_high=12)
v_bp_zero   = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, v,           color='C0', alpha=0.4, label='Original')
ax[0].plot(t, v_bp_causal, color='C1', ls='--', lw=1.2, label='Causal (sosfilt)')
ax[0].set_xlim([0, 2]); ax[0].set_ylabel('Amplitude'); ax[0].legend()
ax[0].set_title('Causal filter — phase delay visible')

ax[1].plot(t, v,         color='C0', alpha=0.4, label='Original')
ax[1].plot(t, v_bp_zero, color='C2', ls='--', lw=1.2, label='Zero-phase (sosfiltfilt)')
ax[1].set_xlim([0, 2]); ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Amplitude'); ax[1].legend()
ax[1].set_title('Zero-phase filter — peaks aligned with original')

plt.tight_layout(); plt.show()

**관찰**:
- 상단 (causal `sosfilt`): 점선 (필터링 결과) 의 봉우리가 원신호 봉우리보다 살짝 **뒤로 밀려** 있다 — 위상 지연이 그대로 남는다.
- 하단 (zero-phase `sosfiltfilt`): 점선의 봉우리가 원신호 봉우리와 **같은 시각** 에 일치 — 앞뒤 두 번 통과로 위상이 정확히 상쇄된다.
- 단, **신호 양 끝 경계** 는 `sosfiltfilt` 의 padding 처리 때문에 zero-phase 에서도 작은 왜곡이 남는다 — 실무에서는 양 끝 일정 구간을 잘라내고 분석한다.
- 사후 분석에서는 "언제 일어났는가" 가 중요하므로 zero-phase 가 기본. 실시간 처리나 cutoff 가 Nyquist 에 매우 가까워 `sosfiltfilt` 가 발산할 때만 causal 로 우회.

---

## 실습 3. High-pass 필터로 저주파 드리프트 제거

DC · 저주파 추세만 제거하고 그 위 모든 고주파 성분은 유지할 때 사용한다.

```python
utils.filtering(x, fs, 'high', f_cut=...)
```

차단 주파수(cutoff) = 5 Hz 로 필터링 후 결과를 비교한다.

In [ ]:
cutoff = 5
v_hp = filtering_zerophase(v, fs, 'high', f_cut=cutoff)

f_raw, A_raw = fft(v, fs)
f_hp,  A_hp  = fft(v_hp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, label='Original')
ax[0].plot(t, v_hp, 'C1',            label=f'Highpass (cutoff={cutoff} Hz)')
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_hp,  A_hp,  'C1',            label='Filtered')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 30])
ax[1].set_ylim([0, 1.3])
ax[1].legend()

fig.tight_layout()
plt.show()

**관찰.** 5 Hz 이하 저주파 성분이 크게 줄고 DC 평균이 0 근처로 맞춰진다. 시간영역에서는 느린 기저선 변동이 사라져 고주파 잡음 및 공진 성분이 그대로 드러난다.

---

## 실습 4. Low-pass 필터로 고주파 잡음 제거

차단 주파수 이하의 저주파 · 추세 성분만 남기고 그 위 고주파 잡음은 제거한다.

```python
utils.filtering(x, fs, 'low', f_cut=...)
```

같은 cutoff = 5 Hz 로 필터링해 실습 3 과 상보적인 결과를 확인한다.

In [ ]:
cutoff = 5
v_lp = filtering_zerophase(v, fs, 'low', f_cut=cutoff)

f_raw, A_raw = fft(v, fs)
f_lp,  A_lp  = fft(v_lp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, label='Original')
ax[0].plot(t, v_lp, 'C1',            label=f'Lowpass (cutoff={cutoff} Hz)')
ax[0].set_xlabel('Time (s)')
ax[0].set_ylabel('y')
ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_lp,  A_lp,  'C1',            label='Filtered')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('|Y|')
ax[1].set_xlim([0, 20])
ax[1].set_ylim([0, 1.3])
ax[1].legend()

fig.tight_layout()
plt.show()

**관찰.** 5 Hz 이상 성분이 거의 사라지고 저주파 추세만 남는다. 실습 3 의 결과와 합치면 원신호가 복원될 것을 기대할 수 있다 — 이 관계를 다음 실습에서 수치로 검증한다.

---

## 실습 5. High + Low ≈ Original (상보성 관찰)

고역통과 + 저역통과 = 원신호 인지 시간 영역에서 검증한다. 영위상 필터 (`filtering_zerophase`) 를 사용한다.

In [ ]:
# HP + LP ≈ Original 확인 (zero-phase 필터)
v_hp = filtering_zerophase(v, fs, 'high', f_cut=5)
v_lp = filtering_zerophase(v, fs, 'low',  f_cut=5)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, v, color='C0', alpha=0.5, lw=1.5, label='Original')
ax.plot(t, v_hp + v_lp, color='C2', ls='--', lw=1.0, label='Recovered (v_hp + v_lp)')
ax.set_xlim([0, 2]); ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude'); ax.legend()
ax.set_title('Zero-phase HP + LP ≈ Original (excluding boundary padding)')

plt.tight_layout(); plt.show()

**관찰**:
- 내부 구간에서 복구 신호 (점선) 가 원신호 (실선) 와 **거의 완벽히 일치** — $|H_{HP}(f)|^2 + |H_{LP}(f)|^2 = 1$ 성질 덕분.
- 단, **신호 양 끝 경계 부분** 은 `sosfiltfilt` 의 padding 처리 때문에 오차가 남는다 — 실무에서는 양 끝을 적당히 무시하고 분석한다.
- 완벽한 시간 영역 상보가 필요하면 `v_hp = v - v_lp` (감산) 같은 단순 방식이 대안.

---

## 실습 6. 응용 신호 — drift + 공진 + 잡음

여기부터 (실습 6 ~ 9) 는 **실측 진동 신호 시나리오** (drift + 공진 + 잡음 합성) 위에서 필터링 워크플로를 시연한다. 본문 §2.4 와 동일한 흐름이다.

먼저 신규 데이터 `data_filtering_application.csv` 를 로드해 (변수 `v` 재할당) raw 신호의 시간 + 스펙트럼을 본다. 데이터는 회전기계 가속도 신호를 모방한 합성 신호 — 저주파 drift (0.5 Hz, 진폭 1.5) + 6 ~ 10 Hz 광대역 공진 + 가우시안 잡음.

In [ ]:
data = np.array(pd.read_csv('./data/data_filtering_application.csv'))
t = data[:, 1]
v = data[:, 2]
print('shape:', data.shape)

f, A = fft(v, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v, 'C0', linewidth=0.6)
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y')
ax[0].set_title('Application signal: drift + resonance + noise')

ax[1].plot(f, A, 'C0')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|')
ax[1].set_title('Single-sided amplitude spectrum (0 to 30 Hz)')

fig.tight_layout(); plt.show()

**관찰.** 스펙트럼에서 세 성분이 확인된다 — (1) 0 ~ 2 Hz drift (~ 0.5 Hz 부근, 진폭 ≈ 1.5), (2) 6 ~ 10 Hz 공진 봉우리 (진폭 ≈ 0.4), (3) 그 위 평평한 잡음 floor (~ 0.05). 이후 실습 7 ~ 9 는 이 세 성분을 단계별로 정리·분해한다.

---

## 실습 7. HPF 로 drift 제거

가장 먼저 떼어낼 것은 저주파 drift. cutoff 3 Hz 의 HPF 를 적용한다.

In [ ]:
v_hp = filtering_zerophase(v, fs, 'high', f_cut=3)

f_raw, A_raw = fft(v, fs)
f_hp,  A_hp  = fft(v_hp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, lw=0.6, label='Original')
ax[0].plot(t, v_hp, 'C1', lw=0.6, label='Highpass (cutoff=3 Hz)')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y'); ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_hp,  A_hp,  'C1',            label='Filtered')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|'); ax[1].legend()

fig.tight_layout(); plt.show()

**관찰.** 시간 영역에서 큰 출렁임 (drift) 이 사라지고 빠른 진동 + 잡음만 남는다. 스펙트럼에서 0.5 Hz 봉우리가 깎이고 6 ~ 10 Hz 공진은 보존된다 — drift 가 제거된 신호 위에서 다음 단계 분석을 시작할 수 있다.

---

## 실습 8. BPF 로 공진 대역 추출

drift 와 잡음을 동시에 떼어내려면 BPF 한 줄로 충분하다. 5 ~ 12 Hz 통과대역을 잡아 공진만 깔끔하게 추출한다.

In [ ]:
v_bp = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)

f_raw, A_raw = fft(v, fs)
f_bp,  A_bp  = fft(v_bp, fs)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(t, v,    'C0', alpha=0.5, lw=0.6, label='Original')
ax[0].plot(t, v_bp, 'C2', lw=0.6, label='Bandpass (5-12 Hz)')
ax[0].set_xlabel('Time (s)'); ax[0].set_ylabel('y'); ax[0].legend()

ax[1].plot(f_raw, A_raw, 'C0', alpha=0.5, label='Original')
ax[1].plot(f_bp,  A_bp,  'C2',            label='Filtered')
ax[1].set_xlim([0, 30]); ax[1].set_ylim([0, 1.7])
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('|Y|'); ax[1].legend()

fig.tight_layout(); plt.show()

**관찰.** 시간 영역에서 깨끗한 6~10 Hz 진동만 남고, 스펙트럼에서 6~10 Hz 봉우리만 솟아 있다. drift 와 잡음이 모두 제거되었다 — 한 줄 BPF 가 HPF + LPF 두 단계의 효과를 동시에 낸다.

---

## 실습 9. 신호 분해 — 3 성분 분리

같은 raw 신호로부터 drift / 공진 / 잡음 세 성분을 모두 분리해 본다. LPF + BPF 두 줄과 한 줄의 산술이면 끝난다.

In [ ]:
drift     = filtering_zerophase(v, fs, 'low',  f_cut=3)
resonance = filtering_zerophase(v, fs, 'band', f_low=5, f_high=12)
noise     = v - drift - resonance

fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
ax[0].plot(t, v,         'C0', lw=0.6); ax[0].set_ylabel('y')
ax[0].set_title('(a) Raw signal (drift + resonance + noise)'); ax[0].set_ylim([-4, 4])

ax[1].plot(t, drift,     'C1', lw=1.0); ax[1].set_ylabel('y')
ax[1].set_title('(b) drift (LPF cutoff 3 Hz)'); ax[1].set_ylim([-4, 4])

ax[2].plot(t, resonance, 'C2', lw=0.6); ax[2].set_ylabel('y')
ax[2].set_title('(c) resonance (BPF 5-12 Hz)'); ax[2].set_ylim([-4, 4])

ax[3].plot(t, noise,     'C3', lw=0.6); ax[3].set_ylabel('y')
ax[3].set_xlabel('Time (s)')
ax[3].set_title('(d) residual noise (raw - drift - resonance)'); ax[3].set_ylim([-4, 4])

fig.tight_layout(); plt.show()

**관찰**:
- (b) drift: 매끈한 0.5 Hz 사인파 (진폭 ≈ 1.5).
- (c) 공진: 빠른 진동 패킷, 진폭이 천천히 변하는 envelope 구조 — Part I Ch4 에서 본격적으로 다룬다.
- (d) 잔여 잡음: 평균 0 부근의 가우시안 분포, 시간적 구조 없음 (백색 잡음).

세 성분이 각각의 물리적 의미 (정렬 미스매치 / 베어링 하우징 모달 / 측정 잡음) 와 명확히 대응한다.

---